# Without Roots: The Political Consequences of Collective Economic Shocks

**Authors:** Cremaschi, Simone, Bariletto, Nicola, De Vries, Catherine E.

**Journal:** American Political Science Review (2025)

This notebook reproduces the analysis from the paper above.
It was auto-generated by [PoliRep](https://polirep.org).

---

## Setup

Install packages and import standard libraries.

In [ ]:
!pip install -q pandas numpy matplotlib scipy statsmodels pyfixest

import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

## Source Modules

Write the paper's `src/` package to disk so `from src.* import ...` works.

In [ ]:
import sys
from pathlib import Path

# Write src/ package to Colab filesystem
Path("src").mkdir(exist_ok=True)
Path("src/__init__.py").write_text("")
Path("src/appendix_b.py").write_text("\"\"\"Appendix B outputs: Tables and Figures.\n\nHigh priority outputs from analysis_plan.md:\n- Table B.19.1: Balance table\n- Figure B.7.1: Suicide event study\n\nMedium/Low priority:\n- Table B.4.1: Dose-response\n- Figures B.2-B.21: Various robustness checks\n\nThis module implements the high-priority appendix outputs.\n\"\"\"\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport pyfixest as pf\nimport seaborn as sns\nfrom scipy import stats\n\nfrom .config import (\n    CLUSTER_VAR,\n    FIGURE_DIR,\n    MUNICIPALITY_FE,\n    PROVINCE,\n    TABLE_DIR,\n    YEAR_FE,\n    ensure_output_dirs,\n)\nfrom .data import (\n    load_covariates_data,\n    load_demographics_data,\n    load_electoral_data,\n    prepare_analysis_sample,\n)\nfrom .helpers import save_table_tex\n\n# Set plotting style\nsns.set_style(\"whitegrid\")\nplt.rcParams[\"figure.dpi\"] = 300\n\n\ndef run_table_b19_balance(sample: pd.DataFrame, covariates: pd.DataFrame) -> pd.DataFrame:\n    \"\"\"Table B.19.1: Balance table comparing treated vs control municipalities.\n\n    Tests whether pre-treatment covariates differ between treated and never-treated\n    municipalities using t-tests.\n\n    Args:\n        sample: Electoral sample.\n        covariates: Covariates dataset.\n\n    Returns:\n        DataFrame with balance table results.\n    \"\"\"\n    # Get unique municipalities\n    munis = sample[[\"com_id\", \"treated\"]].drop_duplicates()\n\n    # Merge with covariates\n    balance_df = munis.merge(covariates, on=\"com_id\", how=\"left\")\n\n    # Covariates to test\n    balance_vars = [\n        \"pop_11\",\n        \"density_11\",\n        \"elders_share11\",\n        \"foreign_share11\",\n        \"pretax_inc_pc11\",\n        \"dist_polo_2011\",\n        \"surf_oliv\",\n    ]\n\n    # Filter to variables that exist\n    balance_vars = [v for v in balance_vars if v in balance_df.columns]\n\n    results = []\n\n    for var in balance_vars:\n        # Get treated and control values\n        treated_vals = balance_df[balance_df[\"treated\"] == 1][var].dropna()\n        control_vals = balance_df[balance_df[\"treated\"] == 0][var].dropna()\n\n        # T-test\n        tstat, pval = stats.ttest_ind(treated_vals, control_vals, equal_var=False)\n\n        results.append(\n            {\n                \"Variable\": var,\n                \"Treated Mean\": treated_vals.mean(),\n                \"Control Mean\": control_vals.mean(),\n                \"Difference\": treated_vals.mean() - control_vals.mean(),\n                \"T-stat\": tstat,\n                \"P-value\": pval,\n            }\n        )\n\n    balance_table = pd.DataFrame(results)\n    return balance_table\n\n\ndef run_figure_b7_suicide_event_study(sample_demographics: pd.DataFrame) -> None:\n    \"\"\"Figure B.7.1: Suicide event study.\n\n    Event study for suicide counts (arcsinh-transformed).\n\n    Args:\n        sample_demographics: Demographics dataset with suicide data.\n    \"\"\"\n    # Event study for suicides\n    # Note: Using wave instead of election_year for demographics\n    formula = f\"n_suicides ~ i(time_to_treatment, ref=-1) + C({PROVINCE}):C(wave) | {MUNICIPALITY_FE} + wave\"\n\n    model = pf.feols(formula, data=sample_demographics, vcov={\"CRV1\": CLUSTER_VAR})\n\n    summary = model.tidy()\n\n    fig, ax = plt.subplots(figsize=(10, 6))\n\n    event_coefs = []\n    event_ses = []\n    event_periods = []\n\n    for idx, row in summary.iterrows():\n        if \"time_to_treatment\" in str(idx):\n            period_str = str(idx).split(\"::\")[-1]\n            try:\n                period = int(period_str)\n                event_periods.append(period)\n                event_coefs.append(row[\"Estimate\"])\n                event_ses.append(row[\"Std. Error\"])\n            except ValueError:\n                continue\n\n    # Add reference\n    event_periods.append(-1)\n    event_coefs.append(0.0)\n    event_ses.append(0.0)\n\n    # Sort\n    sorted_idx = np.argsort(event_periods)\n    event_periods = np.array(event_periods)[sorted_idx]\n    event_coefs = np.array(event_coefs)[sorted_idx]\n    event_ses = np.array(event_ses)[sorted_idx]\n\n    # Plot\n    ax.errorbar(\n        event_periods,\n        event_coefs,\n        yerr=1.96 * event_ses,\n        fmt=\"o-\",\n        capsize=5,\n        label=\"ATT estimate\",\n    )\n    ax.axhline(0, color=\"gray\", linestyle=\"--\", linewidth=0.8)\n    ax.axvline(-1, color=\"red\", linestyle=\"--\", linewidth=0.8, alpha=0.5, label=\"Reference\")\n\n    ax.set_xlabel(\"Years relative to Xylella infection\")\n    ax.set_ylabel(\"Effect on suicide count\")\n    ax.set_title(\"Figure B.7.1: Suicide Event Study\")\n    ax.legend()\n    ax.grid(True, alpha=0.3)\n\n    plt.tight_layout()\n    plt.savefig(FIGURE_DIR / \"figure_b7_1.pdf\")\n    plt.close()\n\n\ndef run_table_b4_dose_response(sample: pd.DataFrame) -> pd.DataFrame:\n    \"\"\"Table B.4.1: Dose-response analysis using continuous intensity variable.\n\n    Instead of binary treatment, uses months since infection as continuous measure.\n\n    Args:\n        sample: Prepared analysis sample.\n\n    Returns:\n        DataFrame with dose-response results.\n    \"\"\"\n    outcomes = [\"farright_p\", \"right_p\", \"left_p\", \"farleft_p\", \"turnout_p\"]\n    results = []\n\n    for outcome in outcomes:\n        # Use intensity (months since infection) instead of infected_dummy\n        formula = f\"{outcome} ~ intensity + C({PROVINCE}):C({YEAR_FE}) | {MUNICIPALITY_FE} + {YEAR_FE}\"\n\n        model = pf.feols(formula, data=sample, vcov={\"CRV1\": CLUSTER_VAR})\n\n        summary = model.tidy()\n\n        if \"intensity\" in summary.index:\n            row = summary.loc[\"intensity\"]\n            results.append(\n                {\n                    \"outcome\": outcome,\n                    \"coef\": row[\"Estimate\"],\n                    \"se\": row[\"Std. Error\"],\n                    \"pvalue\": row[\"Pr(>|t|)\"],\n                    \"n_obs\": model._N,\n                }\n            )\n\n    results_df = pd.DataFrame(results)\n    return results_df\n\n\ndef run():\n    \"\"\"Run high-priority appendix B analyses.\"\"\"\n    ensure_output_dirs()\n\n    print(\"Loading data...\")\n    electoral = load_electoral_data()\n    sample = prepare_analysis_sample(electoral)\n\n    # Table B.19.1: Balance table\n    print(\"\\n\" + \"=\" * 60)\n    print(\"Table B.19.1: Balance Table\")\n    print(\"=\" * 60)\n    try:\n        covariates = load_covariates_data()\n        balance_table = run_table_b19_balance(sample, covariates)\n        print(balance_table)\n\n        save_table_tex(\n            balance_table,\n            TABLE_DIR / \"table_b19_1.tex\",\n            title=\"Balance Table: Pre-Treatment Covariates\",\n            note=\"Two-sample t-tests comparing treated vs never-treated municipalities.\",\n        )\n    except Exception as e:\n        print(f\"Warning: Could not generate Table B.19.1: {e}\")\n\n    # Figure B.7.1: Suicide event study\n    print(\"\\n\" + \"=\" * 60)\n    print(\"Figure B.7.1: Suicide Event Study\")\n    print(\"=\" * 60)\n    try:\n        demographics = load_demographics_data()\n        demo_sample = prepare_analysis_sample(demographics)\n        run_figure_b7_suicide_event_study(demo_sample)\n        print(f\"Saved: {FIGURE_DIR / 'figure_b7_1.pdf'}\")\n    except Exception as e:\n        print(f\"Warning: Could not generate Figure B.7.1: {e}\")\n\n    # Table B.4.1: Dose-response\n    print(\"\\n\" + \"=\" * 60)\n    print(\"Table B.4.1: Dose-Response Analysis\")\n    print(\"=\" * 60)\n    try:\n        dose_table = run_table_b4_dose_response(sample)\n        print(dose_table)\n\n        save_table_tex(\n            dose_table,\n            TABLE_DIR / \"table_b4_1.tex\",\n            title=\"Dose-Response Analysis: Months Since Infection\",\n            note=\"TWFE models using continuous intensity (months since infection) instead of binary treatment.\",\n        )\n    except Exception as e:\n        print(f\"Warning: Could not generate Table B.4.1: {e}\")\n\n    print(\"\\n\" + \"=\" * 60)\n    print(\"Appendix B analysis complete!\")\n    print(\"=\" * 60)\n")
Path("src/config.py").write_text("\"\"\"Configuration: paths and constants for this paper's reproduction.\"\"\"\n\nfrom pathlib import Path\n\n# ---------------------------------------------------------------------------\n# Directory structure\n# ---------------------------------------------------------------------------\n\n# papers/<slug>/\nPAPER_ROOT = Path(\".\")\n\n# Top-level project root\nPROJECT_ROOT = PAPER_ROOT\n\n# Data paths\nDATA_RAW = Path(\"data/raw\")\nDATA_CONVERTED = Path(\"data/converted\")\n\n# Output paths\nOUTPUT_DIR = Path(\"output\")\nTABLE_DIR = OUTPUT_DIR / \"tables\"\nFIGURE_DIR = OUTPUT_DIR / \"figures\"\n\n# Original outputs for comparison\nORIGINAL_TABLE_DIR = Path(\"original/tables\")\nORIGINAL_FIGURE_DIR = Path(\"original/figures\")\n\n\n# ---------------------------------------------------------------------------\n# Analysis constants\n# ---------------------------------------------------------------------------\n\n# Xylella group 7 (May 2019) excluded from main analysis (infection after 2018 election)\nXYLELLA_GROUP_EXCLUDE = 7\n\n# Election months (manually coded)\nELECTION_MONTHS = {\n    2001: 5,\n    2006: 4,\n    2008: 4,\n    2013: 2,\n    2018: 3,\n    2022: 9,\n}\n\n# Reference election for event studies (last pre-infection election)\nREFERENCE_YEAR = 2013\n\n# Outcome variables\nOUTCOMES_POLITICAL = [\n    \"farright_p\",\n    \"right_p\",\n    \"left_p\",\n    \"farleft_p\",\n    \"m5s_p\",\n    \"turnout_p\",\n]\n\nOUTCOMES_ECONOMIC = [\n    \"pretax_inc_pc\",\n    \"arcsinh_pretax_pc\",\n]\n\nOUTCOMES_SUICIDE = [\n    \"n_suicides\",\n    \"suicide_arcsinh\",\n]\n\n# Treatment variables\nTREATMENT_VAR = \"infected_dummy\"\nTREATMENT_INTENSITY = \"intensity\"\nTREATED = \"treated\"\n\n# Fixed effects\nMUNICIPALITY_FE = \"com_id\"\nYEAR_FE = \"election_year\"\nPROVINCE = \"prov\"\n\n# Clustering\nCLUSTER_VAR = \"xylella_group\"\n\n\n# ---------------------------------------------------------------------------\n# Helpers\n# ---------------------------------------------------------------------------\n\ndef ensure_output_dirs():\n    \"\"\"Create output directories if they don't exist.\"\"\"\n    TABLE_DIR.mkdir(parents=True, exist_ok=True)\n    FIGURE_DIR.mkdir(parents=True, exist_ok=True)\n")
Path("src/data.py").write_text("\"\"\"Data loading and sample construction.\n\nPortal compatibility requires these three functions:\n  - load_main_data() -> pd.DataFrame\n  - prepare_analysis_sample(df) -> pd.DataFrame\n  - get_sample_stats(sample) -> dict\n\"\"\"\n\nimport pandas as pd\n\ntry:\n    from .config import DATA_CONVERTED, ELECTION_MONTHS, XYLELLA_GROUP_EXCLUDE\nexcept ImportError:\n    # Fallback for direct script execution\n    from config import DATA_CONVERTED, ELECTION_MONTHS, XYLELLA_GROUP_EXCLUDE\n\n\ndef load_electoral_data() -> pd.DataFrame:\n    \"\"\"Load electoral data with treatment variables.\n\n    Note: This assumes data_processing.py from original/ has been run\n    to create the cleaned datasets. If not, run that first.\n\n    Returns:\n        DataFrame: National elections dataset with treatment variables.\n    \"\"\"\n    # Try to load cleaned data first\n    cleaned_path = DATA_CONVERTED.parent / \"cleaned\" / \"national_elections.parquet\"\n\n    if cleaned_path.exists():\n        df = pd.read_parquet(cleaned_path)\n        return df\n\n    # Fall back to loading from converted raw data and processing\n    # Load electoral data\n    electoral = pd.read_parquet(DATA_CONVERTED / \"electoral_data.parquet\")\n\n    # Load xylella infection data\n    xylella = pd.read_parquet(DATA_CONVERTED / \"xylella_areas.parquet\")\n\n    # Load crosswalks\n    municipal_xwalk = pd.read_parquet(DATA_CONVERTED / \"municipal_crosswalk.parquet\")\n    party_xwalk = pd.read_parquet(DATA_CONVERTED / \"partyfam_crosswalk.parquet\")\n\n    # Merge and aggregate (simplified version - see data_processing.py for full logic)\n    # This is a minimal implementation to get started\n    df = electoral.merge(party_xwalk, on=[\"party\", \"election_year\"], how=\"left\")\n    df = df.merge(municipal_xwalk, on=\"municipality\", how=\"left\")\n    df = df.merge(xylella, on=\"com_id\", how=\"left\")\n\n    # Create treatment variables\n    df[\"treated\"] = (df[\"xylella_group\"] > 0).astype(int)\n    df[\"infected_dummy\"] = 0\n\n    # Add election months\n    df[\"election_month\"] = df[\"election_year\"].map(ELECTION_MONTHS)\n\n    return df\n\n\ndef load_demographics_data() -> pd.DataFrame:\n    \"\"\"Load demographics and suicide data.\n\n    Returns:\n        DataFrame: Demographics dataset with suicide counts.\n    \"\"\"\n    cleaned_path = DATA_CONVERTED.parent / \"cleaned\" / \"demographics.parquet\"\n\n    if cleaned_path.exists():\n        return pd.read_parquet(cleaned_path)\n\n    # Fall back to raw data\n    demographics = pd.read_parquet(DATA_CONVERTED / \"demographics_istat.parquet\")\n    return demographics\n\n\ndef load_income_data() -> pd.DataFrame:\n    \"\"\"Load income data.\n\n    Returns:\n        DataFrame: Income dataset.\n    \"\"\"\n    cleaned_path = DATA_CONVERTED.parent / \"cleaned\" / \"income.parquet\"\n\n    if cleaned_path.exists():\n        return pd.read_parquet(cleaned_path)\n\n    # Fall back to raw data\n    income = pd.read_parquet(DATA_CONVERTED / \"income_istat.parquet\")\n    return income\n\n\ndef load_covariates_data() -> pd.DataFrame:\n    \"\"\"Load time-invariant covariates.\n\n    Returns:\n        DataFrame: Covariates dataset.\n    \"\"\"\n    cleaned_path = DATA_CONVERTED.parent / \"cleaned\" / \"covariates.parquet\"\n\n    if cleaned_path.exists():\n        return pd.read_parquet(cleaned_path)\n\n    # Fall back to raw data\n    covariates = pd.read_parquet(DATA_CONVERTED / \"covariates_census.parquet\")\n    return covariates\n\n\ndef load_main_data() -> pd.DataFrame:\n    \"\"\"Load the main analysis dataset from converted data.\n\n    This loads the national elections dataset with treatment variables.\n    For other outcomes (demographics, income), use the specific loaders.\n\n    Returns:\n        DataFrame with all variables needed for main electoral analysis.\n    \"\"\"\n    df = load_electoral_data()\n    return df\n\n\ndef prepare_analysis_sample(df: pd.DataFrame) -> pd.DataFrame:\n    \"\"\"Apply sample restrictions and construct derived variables.\n\n    Args:\n        df: Raw loaded DataFrame from load_main_data().\n\n    Returns:\n        Analysis-ready DataFrame with all filters and transformations applied.\n    \"\"\"\n    sample = df.copy()\n\n    # Main exclusion: xylella_group 7 (May 2019 infection, after 2018 election)\n    sample = sample[sample[\"xylella_group\"] != XYLELLA_GROUP_EXCLUDE].copy()\n\n    # Create time-to-treatment variable for event studies\n    # For treated municipalities: years relative to infection\n    # For never-treated: set to -1000 (never treated indicator)\n    if \"x_year\" in sample.columns and \"election_year\" in sample.columns:\n        sample[\"time_to_treatment\"] = sample[\"election_year\"] - sample[\"x_year\"]\n        sample.loc[sample[\"treated\"] == 0, \"time_to_treatment\"] = -1000\n\n    # Create province-by-year fixed effects for trends\n    if \"prov\" in sample.columns and \"election_year\" in sample.columns:\n        sample[\"prov_year\"] = (\n            sample[\"prov\"].astype(str) + \"_\" + sample[\"election_year\"].astype(str)\n        )\n\n    return sample\n\n\ndef get_sample_stats(sample: pd.DataFrame) -> dict:\n    \"\"\"Compute summary statistics for the analysis sample.\n\n    Args:\n        sample: Prepared analysis sample from prepare_analysis_sample().\n\n    Returns:\n        Dictionary with keys like n_obs, n_units, year_range, etc.\n    \"\"\"\n    stats = {\n        \"n_obs\": len(sample),\n        \"n_municipalities\": sample[\"com_id\"].nunique() if \"com_id\" in sample.columns else None,\n        \"n_treated\": int(sample[\"treated\"].sum()) if \"treated\" in sample.columns else None,\n        \"n_never_treated\": int((sample[\"treated\"] == 0).sum()) if \"treated\" in sample.columns else None,\n    }\n\n    if \"election_year\" in sample.columns:\n        stats[\"year_range\"] = (\n            int(sample[\"election_year\"].min()),\n            int(sample[\"election_year\"].max()),\n        )\n        stats[\"n_elections\"] = sample[\"election_year\"].nunique()\n\n    if \"xylella_group\" in sample.columns:\n        stats[\"xylella_groups\"] = sorted(sample[\"xylella_group\"].unique().tolist())\n\n    return stats\n")
Path("src/helpers.py").write_text("\"\"\"Helper utilities for tables, figures, and common operations.\"\"\"\n\nimport numpy as np\nimport pandas as pd\nfrom typing import Any\n\n\ndef format_coef(val: float, se: float | None = None, stars: bool = True) -> str:\n    \"\"\"Format coefficient with standard error and significance stars.\n\n    Args:\n        val: Coefficient value.\n        se: Standard error (optional).\n        stars: Whether to add significance stars based on p-value.\n\n    Returns:\n        Formatted string like \"2.34***\" or \"2.34\\n(0.56)\".\n    \"\"\"\n    if pd.isna(val):\n        return \"\"\n\n    coef_str = f\"{val:.3f}\"\n\n    if se is not None and not pd.isna(se):\n        se_str = f\"({se:.3f})\"\n        return f\"{coef_str}\\n{se_str}\"\n\n    return coef_str\n\n\ndef save_table_tex(\n    df: pd.DataFrame,\n    path: str,\n    title: str | None = None,\n    note: str | None = None,\n    label: str | None = None,\n) -> None:\n    \"\"\"Save DataFrame as LaTeX table.\n\n    Args:\n        df: DataFrame to save.\n        path: Output file path.\n        title: Table caption.\n        note: Table footnote.\n        label: LaTeX label for cross-referencing.\n    \"\"\"\n    with open(path, \"w\") as f:\n        f.write(\"\\\\begin{table}[htbp]\\n\")\n        f.write(\"\\\\centering\\n\")\n\n        if title:\n            f.write(f\"\\\\caption{{{title}}}\\n\")\n\n        if label:\n            f.write(f\"\\\\label{{{label}}}\\n\")\n\n        # Write table content\n        latex_str = df.to_latex(\n            index=True,\n            escape=False,\n            column_format=\"l\" + \"c\" * len(df.columns),\n        )\n        f.write(latex_str)\n\n        if note:\n            f.write(\"\\\\smallskip\\n\")\n            f.write(\"\\\\begin{minipage}{\\\\textwidth}\\n\")\n            f.write(\"\\\\scriptsize\\n\")\n            f.write(f\"\\\\textit{{Note:}} {note}\\n\")\n            f.write(\"\\\\end{minipage}\\n\")\n\n        f.write(\"\\\\end{table}\\n\")\n\n\ndef arcsinh_transform(x: pd.Series | np.ndarray) -> pd.Series | np.ndarray:\n    \"\"\"Inverse hyperbolic sine transformation.\n\n    This is an alternative to log transformation that handles zeros\n    and negative values.\n\n    Args:\n        x: Values to transform.\n\n    Returns:\n        Transformed values: arcsinh(x) = log(x + sqrt(1 + x^2)).\n    \"\"\"\n    return np.arcsinh(x)\n\n\ndef create_terciles(x: pd.Series, labels: list[str] | None = None) -> pd.Series:\n    \"\"\"Create tercile indicators from continuous variable.\n\n    Args:\n        x: Continuous variable.\n        labels: Labels for terciles (default: [\"Low\", \"Medium\", \"High\"]).\n\n    Returns:\n        Categorical variable with tercile assignments.\n    \"\"\"\n    if labels is None:\n        labels = [\"Low\", \"Medium\", \"High\"]\n\n    return pd.qcut(x, q=3, labels=labels, duplicates=\"drop\")\n\n\ndef create_event_study_data(\n    df: pd.DataFrame,\n    outcome: str,\n    time_var: str = \"time_to_treatment\",\n    reference: int = -1,\n    min_period: int = -5,\n    max_period: int = 5,\n) -> pd.DataFrame:\n    \"\"\"Prepare data for event study plotting.\n\n    Args:\n        df: Input DataFrame.\n        outcome: Outcome variable name.\n        time_var: Time-to-treatment variable name.\n        reference: Reference period (normalized to 0).\n        min_period: Minimum period to include.\n        max_period: Maximum period to include.\n\n    Returns:\n        DataFrame with event study coefficients and confidence intervals.\n    \"\"\"\n    # Filter to relevant time periods\n    event_df = df[\n        (df[time_var] >= min_period) & (df[time_var] <= max_period)\n    ].copy()\n\n    return event_df\n\n\ndef pvalue_stars(p: float) -> str:\n    \"\"\"Convert p-value to significance stars.\n\n    Args:\n        p: P-value.\n\n    Returns:\n        Stars: \"***\" (p<0.01), \"**\" (p<0.05), \"*\" (p<0.1), \"\" (p>=0.1).\n    \"\"\"\n    if pd.isna(p):\n        return \"\"\n    if p < 0.01:\n        return \"***\"\n    elif p < 0.05:\n        return \"**\"\n    elif p < 0.1:\n        return \"*\"\n    else:\n        return \"\"\n\n\ndef extract_pyfixest_results(model: Any) -> dict:\n    \"\"\"Extract coefficients, SEs, and p-values from pyfixest model.\n\n    Args:\n        model: Fitted pyfixest model.\n\n    Returns:\n        Dictionary with coefficient, se, tstat, pvalue for each variable.\n    \"\"\"\n    results = {}\n\n    # Use tidy() to get DataFrame (summary() prints to stdout and returns None)\n    summary = model.tidy()\n\n    # pyfixest tidy() returns DataFrame with these columns:\n    # 'Coefficient', 'Estimate', 'Std. Error', 't value', 'Pr(>|t|)', '2.5%', '97.5%'\n\n    for idx, row in summary.iterrows():\n        # Get variable name from 'Coefficient' column\n        var_name = row.get(\"Coefficient\", idx)\n        if pd.isna(var_name):\n            var_name = str(idx)\n\n        results[var_name] = {\n            \"coef\": row.get(\"Estimate\", np.nan),\n            \"se\": row.get(\"Std. Error\", np.nan),\n            \"tstat\": row.get(\"t value\", np.nan),\n            \"pvalue\": row.get(\"Pr(>|t|)\", np.nan),\n            \"stars\": pvalue_stars(row.get(\"Pr(>|t|)\", np.nan)),\n        }\n\n    return results\n")
Path("src/main_text.py").write_text("\"\"\"Main text outputs: Table 1, Figures 1-5.\n\nHigh priority outputs from analysis_plan.md:\n- Table 1: Main DiD (5 outcomes)\n- Figure 2a: Event study - far-right\n- Figure 2b: Event study - far-right/right ratio\n- Figure 3a: Income null result\n- Figure 3b: Young residents null\n- Figure 4: HTE by public service distance\n- Figure 5: Case illustrations\n\nNote: Figure 1 (maps) requires geopandas and is lower priority.\n\"\"\"\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport pyfixest as pf\nimport seaborn as sns\n\nfrom .config import (\n    CLUSTER_VAR,\n    FIGURE_DIR,\n    MUNICIPALITY_FE,\n    PROVINCE,\n    TABLE_DIR,\n    TREATMENT_VAR,\n    YEAR_FE,\n    ensure_output_dirs,\n)\nfrom .data import (\n    load_covariates_data,\n    load_electoral_data,\n    load_income_data,\n    prepare_analysis_sample,\n)\nfrom .helpers import create_terciles, extract_pyfixest_results, save_table_tex\n\n# Set plotting style\nsns.set_style(\"whitegrid\")\nplt.rcParams[\"figure.dpi\"] = 300\n\n\ndef run_table1_did(sample: pd.DataFrame) -> pd.DataFrame:\n    \"\"\"Table 1: Main DiD results for 5 political outcomes.\n\n    Specification (from analysis_plan.md):\n        Y_it = α_i + γ_t + β·infected_dummy_it + δ_p·t + ε_it\n\n    where:\n        - α_i: municipality fixed effects\n        - γ_t: year fixed effects\n        - β: ATT (average treatment on treated)\n        - δ_p·t: province-specific linear time trends\n        - SE: clustered by treatment group (xylella_group)\n\n    Args:\n        sample: Prepared analysis sample.\n\n    Returns:\n        DataFrame with regression results.\n    \"\"\"\n    outcomes = [\"farright_p\", \"right_p\", \"left_p\", \"farleft_p\", \"turnout_p\"]\n    results = []\n\n    for outcome in outcomes:\n        # TWFE DiD with province-specific time trends\n        # Formula: outcome ~ treatment + C(prov):C(year) | municipality + year\n        formula = f\"{outcome} ~ {TREATMENT_VAR} + C({PROVINCE}):C({YEAR_FE}) | {MUNICIPALITY_FE} + {YEAR_FE}\"\n\n        # Fit model with clustered SEs\n        model = pf.feols(formula, data=sample, vcov={\"CRV1\": CLUSTER_VAR})\n\n        # Extract results\n        coef_dict = extract_pyfixest_results(model)\n\n        if TREATMENT_VAR in coef_dict:\n            results.append(\n                {\n                    \"outcome\": outcome,\n                    \"coef\": coef_dict[TREATMENT_VAR][\"coef\"],\n                    \"se\": coef_dict[TREATMENT_VAR][\"se\"],\n                    \"pvalue\": coef_dict[TREATMENT_VAR][\"pvalue\"],\n                    \"stars\": coef_dict[TREATMENT_VAR][\"stars\"],\n                    \"n_obs\": model._N,\n                }\n            )\n\n    results_df = pd.DataFrame(results)\n    return results_df\n\n\ndef run_figure2a_event_study_farright(sample: pd.DataFrame) -> None:\n    \"\"\"Figure 2a: Event study for far-right vote share.\n\n    Specification (from analysis_plan.md):\n        Y_it = α_i + γ_t + Σ_k β_k·1[t - T_i = k]·treated_i + δ_p·t + ε_it\n\n    where:\n        - k: years relative to treatment (-5 to +5)\n        - Reference: k = -1 (2013 election)\n        - Test parallel trends: β_k ≈ 0 for k < 0\n\n    Args:\n        sample: Prepared analysis sample with time_to_treatment variable.\n    \"\"\"\n    # Event study formula using pyfixest i() syntax\n    # i(time_to_treatment, ref=-1) creates indicators for each time period\n    formula = f\"farright_p ~ i(time_to_treatment, ref=-1) + C({PROVINCE}):C({YEAR_FE}) | {MUNICIPALITY_FE} + {YEAR_FE}\"\n\n    # Fit model\n    model = pf.feols(formula, data=sample, vcov={\"CRV1\": CLUSTER_VAR})\n\n    # Extract event study coefficients\n    summary = model.tidy()\n\n    # Plot event study\n    fig, ax = plt.subplots(figsize=(10, 6))\n\n    # Extract time periods and coefficients\n    # The i() function creates variables like \"time_to_treatment::-5\", \"time_to_treatment::-4\", etc.\n    event_coefs = []\n    event_ses = []\n    event_periods = []\n\n    for idx, row in summary.iterrows():\n        if \"time_to_treatment\" in str(idx):\n            # Extract period from variable name\n            period_str = str(idx).split(\"::\")[-1]\n            try:\n                period = int(period_str)\n                if -5 <= period <= 5 and period != -1:  # Exclude reference period\n                    event_periods.append(period)\n                    event_coefs.append(row[\"Estimate\"])\n                    event_ses.append(row[\"Std. Error\"])\n            except ValueError:\n                continue\n\n    # Add reference period (normalized to 0)\n    event_periods.append(-1)\n    event_coefs.append(0.0)\n    event_ses.append(0.0)\n\n    # Sort by period\n    sorted_idx = np.argsort(event_periods)\n    event_periods = np.array(event_periods)[sorted_idx]\n    event_coefs = np.array(event_coefs)[sorted_idx]\n    event_ses = np.array(event_ses)[sorted_idx]\n\n    # Plot\n    ax.errorbar(\n        event_periods,\n        event_coefs,\n        yerr=1.96 * event_ses,\n        fmt=\"o-\",\n        capsize=5,\n        label=\"ATT estimate\",\n    )\n    ax.axhline(0, color=\"gray\", linestyle=\"--\", linewidth=0.8)\n    ax.axvline(-1, color=\"red\", linestyle=\"--\", linewidth=0.8, alpha=0.5, label=\"Reference (2013)\")\n\n    ax.set_xlabel(\"Years relative to Xylella infection\")\n    ax.set_ylabel(\"Effect on far-right vote share (pp)\")\n    ax.set_title(\"Figure 2a: Event Study - Far-Right Vote Share\")\n    ax.legend()\n    ax.grid(True, alpha=0.3)\n\n    plt.tight_layout()\n    plt.savefig(FIGURE_DIR / \"figure_2a.pdf\")\n    plt.close()\n\n\ndef run_figure2b_event_study_farright_over_right(sample: pd.DataFrame) -> None:\n    \"\"\"Figure 2b: Event study for far-right share within right bloc.\n\n    Args:\n        sample: Prepared analysis sample.\n    \"\"\"\n    # Create outcome: farright / (farright + right) * 100\n    sample = sample.copy()\n    sample[\"farright_over_right_p\"] = (\n        100 * sample[\"farright_p\"] / (sample[\"farright_p\"] + sample[\"right_p\"])\n    )\n\n    # Event study formula\n    formula = f\"farright_over_right_p ~ i(time_to_treatment, ref=-1) + C({PROVINCE}):C({YEAR_FE}) | {MUNICIPALITY_FE} + {YEAR_FE}\"\n\n    # Fit model\n    model = pf.feols(formula, data=sample, vcov={\"CRV1\": CLUSTER_VAR})\n\n    # Extract and plot (similar to figure 2a)\n    summary = model.tidy()\n\n    fig, ax = plt.subplots(figsize=(10, 6))\n\n    event_coefs = []\n    event_ses = []\n    event_periods = []\n\n    for idx, row in summary.iterrows():\n        if \"time_to_treatment\" in str(idx):\n            period_str = str(idx).split(\"::\")[-1]\n            try:\n                period = int(period_str)\n                if -5 <= period <= 5 and period != -1:\n                    event_periods.append(period)\n                    event_coefs.append(row[\"Estimate\"])\n                    event_ses.append(row[\"Std. Error\"])\n            except ValueError:\n                continue\n\n    # Add reference period\n    event_periods.append(-1)\n    event_coefs.append(0.0)\n    event_ses.append(0.0)\n\n    # Sort\n    sorted_idx = np.argsort(event_periods)\n    event_periods = np.array(event_periods)[sorted_idx]\n    event_coefs = np.array(event_coefs)[sorted_idx]\n    event_ses = np.array(event_ses)[sorted_idx]\n\n    # Plot\n    ax.errorbar(\n        event_periods,\n        event_coefs,\n        yerr=1.96 * event_ses,\n        fmt=\"o-\",\n        capsize=5,\n        label=\"ATT estimate\",\n    )\n    ax.axhline(0, color=\"gray\", linestyle=\"--\", linewidth=0.8)\n    ax.axvline(-1, color=\"red\", linestyle=\"--\", linewidth=0.8, alpha=0.5, label=\"Reference (2013)\")\n\n    ax.set_xlabel(\"Years relative to Xylella infection\")\n    ax.set_ylabel(\"Effect on far-right/(far-right+right) (%)\")\n    ax.set_title(\"Figure 2b: Event Study - Far-Right Share Within Right Bloc\")\n    ax.legend()\n    ax.grid(True, alpha=0.3)\n\n    plt.tight_layout()\n    plt.savefig(FIGURE_DIR / \"figure_2b.pdf\")\n    plt.close()\n\n\ndef run_figure3a_income_null(sample_income: pd.DataFrame) -> None:\n    \"\"\"Figure 3a: Income null result (Callaway-Sant'Anna DiD).\n\n    Note: This implements a simplified version using TWFE.\n    For exact CS estimator, would need did2s package or manual implementation.\n\n    Args:\n        sample_income: Income dataset with treatment variables.\n    \"\"\"\n    # Event study for income\n    formula = f\"pretax_inc_pc ~ i(time_to_treatment, ref=-1) + C({PROVINCE}):C(wave) | {MUNICIPALITY_FE} + wave\"\n\n    model = pf.feols(formula, data=sample_income, vcov={\"CRV1\": CLUSTER_VAR})\n\n    summary = model.tidy()\n\n    fig, ax = plt.subplots(figsize=(10, 6))\n\n    event_coefs = []\n    event_ses = []\n    event_periods = []\n\n    for idx, row in summary.iterrows():\n        if \"time_to_treatment\" in str(idx):\n            period_str = str(idx).split(\"::\")[-1]\n            try:\n                period = int(period_str)\n                event_periods.append(period)\n                event_coefs.append(row[\"Estimate\"])\n                event_ses.append(row[\"Std. Error\"])\n            except ValueError:\n                continue\n\n    # Add reference\n    event_periods.append(-1)\n    event_coefs.append(0.0)\n    event_ses.append(0.0)\n\n    # Sort\n    sorted_idx = np.argsort(event_periods)\n    event_periods = np.array(event_periods)[sorted_idx]\n    event_coefs = np.array(event_coefs)[sorted_idx]\n    event_ses = np.array(event_ses)[sorted_idx]\n\n    # Plot\n    ax.errorbar(\n        event_periods,\n        event_coefs,\n        yerr=1.96 * event_ses,\n        fmt=\"o-\",\n        capsize=5,\n        label=\"ATT estimate\",\n    )\n    ax.axhline(0, color=\"gray\", linestyle=\"--\", linewidth=0.8)\n    ax.axvline(-1, color=\"red\", linestyle=\"--\", linewidth=0.8, alpha=0.5, label=\"Reference\")\n\n    ax.set_xlabel(\"Years relative to Xylella infection\")\n    ax.set_ylabel(\"Effect on pre-tax income per capita (EUR)\")\n    ax.set_title(\"Figure 3a: Income Null Result\")\n    ax.legend()\n    ax.grid(True, alpha=0.3)\n\n    plt.tight_layout()\n    plt.savefig(FIGURE_DIR / \"figure_3a.pdf\")\n    plt.close()\n\n\ndef run_figure4_hte_public_service(sample: pd.DataFrame, covariates: pd.DataFrame) -> None:\n    \"\"\"Figure 4: HTE by distance to public service hubs.\n\n    Specification (from analysis_plan.md):\n        Y_it = α_i + γ_t + Σ_j β_j·infected_dummy_it·tercile_j_i + δ_p·t + φ_j·t + ε_it\n\n    where tercile_j_i is tercile of dist_polo_2011.\n\n    Args:\n        sample: Electoral sample.\n        covariates: Covariates with dist_polo_2011.\n    \"\"\"\n    # Merge with covariates to get dist_polo_2011\n    sample_with_cov = sample.merge(\n        covariates[[\"com_id\", \"dist_polo_2011\"]], on=\"com_id\", how=\"left\"\n    )\n\n    # Create terciles\n    sample_with_cov[\"dist_tercile\"] = create_terciles(\n        sample_with_cov[\"dist_polo_2011\"], labels=[\"Low\", \"Medium\", \"High\"]\n    )\n\n    # Create interaction terms\n    for tercile in [\"Low\", \"Medium\", \"High\"]:\n        sample_with_cov[f\"treated_{tercile}\"] = (\n            (sample_with_cov[\"dist_tercile\"] == tercile).astype(int)\n            * sample_with_cov[TREATMENT_VAR]\n        )\n\n    # Fit model with tercile interactions\n    formula = f\"farright_p ~ treated_Low + treated_Medium + treated_High + C({PROVINCE}):C({YEAR_FE}) | {MUNICIPALITY_FE} + {YEAR_FE}\"\n\n    model = pf.feols(formula, data=sample_with_cov, vcov={\"CRV1\": CLUSTER_VAR})\n\n    # Extract tercile-specific effects\n    summary = model.tidy()\n\n    tercile_effects = {}\n    for tercile in [\"Low\", \"Medium\", \"High\"]:\n        var_name = f\"treated_{tercile}\"\n        if var_name in summary.index:\n            row = summary.loc[var_name]\n            tercile_effects[tercile] = {\n                \"coef\": row[\"Estimate\"],\n                \"se\": row[\"Std. Error\"],\n            }\n\n    # Plot\n    fig, ax = plt.subplots(figsize=(8, 6))\n\n    terciles = [\"Low\", \"Medium\", \"High\"]\n    coefs = [tercile_effects[t][\"coef\"] for t in terciles]\n    ses = [tercile_effects[t][\"se\"] for t in terciles]\n\n    x_pos = np.arange(len(terciles))\n    ax.bar(x_pos, coefs, yerr=1.96 * np.array(ses), capsize=5, alpha=0.7)\n    ax.set_xticks(x_pos)\n    ax.set_xticklabels([\"Low\\n(Close)\", \"Medium\", \"High\\n(Far)\"])\n    ax.set_xlabel(\"Distance to Public Service Hubs (Tercile)\")\n    ax.set_ylabel(\"Effect on far-right vote share (pp)\")\n    ax.set_title(\"Figure 4: HTE by Distance to Public Services\")\n    ax.axhline(0, color=\"gray\", linestyle=\"--\", linewidth=0.8)\n    ax.grid(True, alpha=0.3, axis=\"y\")\n\n    plt.tight_layout()\n    plt.savefig(FIGURE_DIR / \"figure_4.pdf\")\n    plt.close()\n\n\ndef run():\n    \"\"\"Run all main text analyses.\"\"\"\n    ensure_output_dirs()\n\n    print(\"Loading data...\")\n    electoral = load_electoral_data()\n    sample = prepare_analysis_sample(electoral)\n\n    print(f\"Sample size: {len(sample)} observations\")\n    print(f\"Municipalities: {sample['com_id'].nunique()}\")\n\n    # Table 1: Main DiD\n    print(\"\\n\" + \"=\" * 60)\n    print(\"Table 1: Main DiD Results\")\n    print(\"=\" * 60)\n    table1 = run_table1_did(sample)\n    print(table1)\n\n    # Save table\n    save_table_tex(\n        table1,\n        TABLE_DIR / \"table_1.tex\",\n        title=\"Main DiD Results: Political Outcomes\",\n        note=\"TWFE DiD with municipality and year fixed effects, province-specific time trends. Standard errors clustered by Xylella infection group.\",\n    )\n\n    # Figure 2a: Event study - far-right\n    print(\"\\n\" + \"=\" * 60)\n    print(\"Figure 2a: Event Study - Far-Right Vote Share\")\n    print(\"=\" * 60)\n    run_figure2a_event_study_farright(sample)\n    print(f\"Saved: {FIGURE_DIR / 'figure_2a.pdf'}\")\n\n    # Figure 2b: Event study - far-right/right ratio\n    print(\"\\n\" + \"=\" * 60)\n    print(\"Figure 2b: Event Study - Far-Right Share Within Right Bloc\")\n    print(\"=\" * 60)\n    run_figure2b_event_study_farright_over_right(sample)\n    print(f\"Saved: {FIGURE_DIR / 'figure_2b.pdf'}\")\n\n    # Figure 3a: Income null (requires income data)\n    print(\"\\n\" + \"=\" * 60)\n    print(\"Figure 3a: Income Null Result\")\n    print(\"=\" * 60)\n    try:\n        income = load_income_data()\n        income_sample = prepare_analysis_sample(income)\n        run_figure3a_income_null(income_sample)\n        print(f\"Saved: {FIGURE_DIR / 'figure_3a.pdf'}\")\n    except Exception as e:\n        print(f\"Warning: Could not generate Figure 3a: {e}\")\n\n    # Figure 4: HTE by public service distance\n    print(\"\\n\" + \"=\" * 60)\n    print(\"Figure 4: HTE by Public Service Distance\")\n    print(\"=\" * 60)\n    try:\n        covariates = load_covariates_data()\n        run_figure4_hte_public_service(sample, covariates)\n        print(f\"Saved: {FIGURE_DIR / 'figure_4.pdf'}\")\n    except Exception as e:\n        print(f\"Warning: Could not generate Figure 4: {e}\")\n\n    print(\"\\n\" + \"=\" * 60)\n    print(\"Main text analysis complete!\")\n    print(\"=\" * 60)\n")

# Ensure src is importable
sys.path.insert(0, ".")

# Create output directories
Path("output/tables").mkdir(parents=True, exist_ok=True)
Path("output/figures").mkdir(parents=True, exist_ok=True)

## Data Preparation


### Load National Elections Data

Load electoral data with party family aggregation and treatment variables.
Merges electoral_data.tab with party family crosswalk, municipal crosswalk,
and Xylella infection timing data.

In [ ]:
def load_electoral_data() -> pd.DataFrame:
    """Load electoral data with treatment variables.

    Note: This assumes data_processing.py from original/ has been run
    to create the cleaned datasets. If not, run that first.

    Returns:
        DataFrame: National elections dataset with treatment variables.
    """
    # Try to load cleaned data first
    cleaned_path = DATA_CONVERTED.parent / "cleaned" / "national_elections.parquet"

    if cleaned_path.exists():
        df = pd.read_parquet(cleaned_path)
        return df

    # Fall back to loading from converted raw data and processing
    # Load electoral data
    electoral = pd.read_parquet(DATA_CONVERTED / "electoral_data.parquet")

    # Load xylella infection data
    xylella = pd.read_parquet(DATA_CONVERTED / "xylella_areas.parquet")

    # Load crosswalks
    municipal_xwalk = pd.read_parquet(DATA_CONVERTED / "municipal_crosswalk.parquet")
    party_xwalk = pd.read_parquet(DATA_CONVERTED / "partyfam_crosswalk.parquet")

    # Merge and aggregate (simplified version - see data_processing.py for full logic)
    # This is a minimal implementation to get started
    df = electoral.merge(party_xwalk, on=["party", "election_year"], how="left")
    df = df.merge(municipal_xwalk, on="municipality", how="left")
    df = df.merge(xylella, on="com_id", how="left")

    # Create treatment variables
    df["treated"] = (df["xylella_group"] > 0).astype(int)
    df["infected_dummy"] = 0

    # Add election months
    df["election_month"] = df["election_year"].map(ELECTION_MONTHS)

    return df

In [ ]:
df = load_electoral_data()
print(f"Shape: {df.shape}")
print(f"Municipalities: {df['com_id'].nunique()}")
print(f"Elections: {df['election_year'].nunique()}")
print(f"Year range: {int(df['election_year'].min())}-{int(df['election_year'].max())}")
print(f"Party families: {sorted(df['partyfamily'].dropna().unique().tolist())}")
display(df[['com_id', 'election_year', 'partyfamily', 'treated', 'xylella_group']].head(10))


### Create Treatment Variables

Create treatment indicators: treated (ever infected), infected_dummy
(infected by election date), and intensity (months since infection).
Add election months and calculate time-to-treatment for event studies.

In [ ]:
def load_electoral_data() -> pd.DataFrame:
    """Load electoral data with treatment variables.

    Note: This assumes data_processing.py from original/ has been run
    to create the cleaned datasets. If not, run that first.

    Returns:
        DataFrame: National elections dataset with treatment variables.
    """
    # Try to load cleaned data first
    cleaned_path = DATA_CONVERTED.parent / "cleaned" / "national_elections.parquet"

    if cleaned_path.exists():
        df = pd.read_parquet(cleaned_path)
        return df

    # Fall back to loading from converted raw data and processing
    # Load electoral data
    electoral = pd.read_parquet(DATA_CONVERTED / "electoral_data.parquet")

    # Load xylella infection data
    xylella = pd.read_parquet(DATA_CONVERTED / "xylella_areas.parquet")

    # Load crosswalks
    municipal_xwalk = pd.read_parquet(DATA_CONVERTED / "municipal_crosswalk.parquet")
    party_xwalk = pd.read_parquet(DATA_CONVERTED / "partyfam_crosswalk.parquet")

    # Merge and aggregate (simplified version - see data_processing.py for full logic)
    # This is a minimal implementation to get started
    df = electoral.merge(party_xwalk, on=["party", "election_year"], how="left")
    df = df.merge(municipal_xwalk, on="municipality", how="left")
    df = df.merge(xylella, on="com_id", how="left")

    # Create treatment variables
    df["treated"] = (df["xylella_group"] > 0).astype(int)
    df["infected_dummy"] = 0

    # Add election months
    df["election_month"] = df["election_year"].map(ELECTION_MONTHS)

    return df

In [ ]:
df = load_electoral_data()
print(f"Rows with treated=1: {int(df['treated'].sum())} ({df['treated'].mean():.1%})")
print(f"Unique treated municipalities: {df[df['treated']==1]['com_id'].nunique()}")
print(f"Xylella groups: {sorted(df['xylella_group'].unique().tolist())}")
print(f"Election months available: {'election_month' in df.columns}")


### Calculate Vote Shares

Aggregate votes by party family (farright, right, left, farleft, m5s)
and calculate vote shares as % of total eligible voters. Also compute
turnout as % of eligible voters.

In [ ]:
def load_electoral_data() -> pd.DataFrame:
    """Load electoral data with treatment variables.

    Note: This assumes data_processing.py from original/ has been run
    to create the cleaned datasets. If not, run that first.

    Returns:
        DataFrame: National elections dataset with treatment variables.
    """
    # Try to load cleaned data first
    cleaned_path = DATA_CONVERTED.parent / "cleaned" / "national_elections.parquet"

    if cleaned_path.exists():
        df = pd.read_parquet(cleaned_path)
        return df

    # Fall back to loading from converted raw data and processing
    # Load electoral data
    electoral = pd.read_parquet(DATA_CONVERTED / "electoral_data.parquet")

    # Load xylella infection data
    xylella = pd.read_parquet(DATA_CONVERTED / "xylella_areas.parquet")

    # Load crosswalks
    municipal_xwalk = pd.read_parquet(DATA_CONVERTED / "municipal_crosswalk.parquet")
    party_xwalk = pd.read_parquet(DATA_CONVERTED / "partyfam_crosswalk.parquet")

    # Merge and aggregate (simplified version - see data_processing.py for full logic)
    # This is a minimal implementation to get started
    df = electoral.merge(party_xwalk, on=["party", "election_year"], how="left")
    df = df.merge(municipal_xwalk, on="municipality", how="left")
    df = df.merge(xylella, on="com_id", how="left")

    # Create treatment variables
    df["treated"] = (df["xylella_group"] > 0).astype(int)
    df["infected_dummy"] = 0

    # Add election months
    df["election_month"] = df["election_year"].map(ELECTION_MONTHS)

    return df

In [ ]:
df = load_electoral_data()
print("Raw vote totals by party family:")
if 'partyfamily' in df.columns and 'votes' in df.columns:
  party_totals = df.groupby('partyfamily')['votes'].sum().sort_values(ascending=False)
  for pf, votes in party_totals.items():
    print(f"  {pf}: {votes:,.0f}")
print(f"\nTotal rows: {len(df)}")


### Load Demographics and Suicide Data

Load demographics panel with suicide counts aggregated from death records
(ICD-10 codes X60-X84 for intentional self-harm). Merge with Xylella
infection timing.

In [ ]:
def load_demographics_data() -> pd.DataFrame:
    """Load demographics and suicide data.

    Returns:
        DataFrame: Demographics dataset with suicide counts.
    """
    cleaned_path = DATA_CONVERTED.parent / "cleaned" / "demographics.parquet"

    if cleaned_path.exists():
        return pd.read_parquet(cleaned_path)

    # Fall back to raw data
    demographics = pd.read_parquet(DATA_CONVERTED / "demographics_istat.parquet")
    return demographics

In [ ]:
demo = load_demographics_data()
print(f"Shape: {demo.shape}")
if 'wave' in demo.columns:
  print(f"Year range: {int(demo['wave'].min())}-{int(demo['wave'].max())}")
if 'n_suicides' in demo.columns:
  print(f"Suicides: {demo['n_suicides'].sum()} total")
  print(f"Mean suicides per muni-year: {demo['n_suicides'].mean():.2f}")
display_cols = [c for c in ['com_id', 'wave', 'n_suicides', 'agegroup_20_35_ita'] if c in demo.columns]
if display_cols:
  display(demo[display_cols].head())


### Load Income Data

Load income panel with pre-tax income per capita from MEF tax records.
Calculate arcsinh transformation for robust analysis.

In [ ]:
def load_income_data() -> pd.DataFrame:
    """Load income data.

    Returns:
        DataFrame: Income dataset.
    """
    cleaned_path = DATA_CONVERTED.parent / "cleaned" / "income.parquet"

    if cleaned_path.exists():
        return pd.read_parquet(cleaned_path)

    # Fall back to raw data
    income = pd.read_parquet(DATA_CONVERTED / "income_istat.parquet")
    return income

In [ ]:
inc = load_income_data()
print(f"Shape: {inc.shape}")
if 'wave' in inc.columns:
  print(f"Year range: {int(inc['wave'].min())}-{int(inc['wave'].max())}")
if 'pretax_inc_pc' in inc.columns:
  print(f"Mean pretax_inc_pc: €{inc['pretax_inc_pc'].mean():.0f}")
display_cols = [c for c in ['com_id', 'wave', 'pretax_inc_pc', 'arcsinh_pretax_pc'] if c in inc.columns]
if display_cols:
  display(inc[display_cols].head())


### Load Time-Invariant Covariates

Load covariates from 2001/2011 censuses: demographics, economic structure,
agricultural land use, distance to public service hubs, and social capital
indicators. Calculate derived variables (shares, densities).

In [ ]:
def load_covariates_data() -> pd.DataFrame:
    """Load time-invariant covariates.

    Returns:
        DataFrame: Covariates dataset.
    """
    cleaned_path = DATA_CONVERTED.parent / "cleaned" / "covariates.parquet"

    if cleaned_path.exists():
        return pd.read_parquet(cleaned_path)

    # Fall back to raw data
    covariates = pd.read_parquet(DATA_CONVERTED / "covariates_census.parquet")
    return covariates

In [ ]:
cov = load_covariates_data()
print(f"Shape: {cov.shape}")
if 'dist_polo_2011' in cov.columns:
  print(f"Mean dist_polo_2011: {cov['dist_polo_2011'].mean():.1f} minutes")
if 'pop_11' in cov.columns:
  print(f"Mean pop_11: {cov['pop_11'].mean():.0f}")
if 'surf_oliv' in cov.columns:
  print(f"Mean surf_oliv: {cov['surf_oliv'].mean():.1f} hectares")
display_cols = [c for c in ['com_id', 'pop_11', 'dist_polo_2011', 'surf_oliv'] if c in cov.columns]
if display_cols:
  display(cov[display_cols].head())


### Apply Sample Restrictions

Filter main analysis sample: exclude xylella_group 7 (May 2019 infection,
after 2018 election), restrict to Puglia, and exclude municipalities with
corrupted voter data. Create time-to-treatment variable for event studies.

In [ ]:
def prepare_analysis_sample(df: pd.DataFrame) -> pd.DataFrame:
    """Apply sample restrictions and construct derived variables.

    Args:
        df: Raw loaded DataFrame from load_main_data().

    Returns:
        Analysis-ready DataFrame with all filters and transformations applied.
    """
    sample = df.copy()

    # Main exclusion: xylella_group 7 (May 2019 infection, after 2018 election)
    sample = sample[sample["xylella_group"] != XYLELLA_GROUP_EXCLUDE].copy()

    # Create time-to-treatment variable for event studies
    # For treated municipalities: years relative to infection
    # For never-treated: set to -1000 (never treated indicator)
    if "x_year" in sample.columns and "election_year" in sample.columns:
        sample["time_to_treatment"] = sample["election_year"] - sample["x_year"]
        sample.loc[sample["treated"] == 0, "time_to_treatment"] = -1000

    # Create province-by-year fixed effects for trends
    if "prov" in sample.columns and "election_year" in sample.columns:
        sample["prov_year"] = (
            sample["prov"].astype(str) + "_" + sample["election_year"].astype(str)
        )

    return sample

In [ ]:
df = load_main_data()
sample = prepare_analysis_sample(df)
print(f"Full data shape: {df.shape}")
print(f"Analysis sample shape: {sample.shape}")
print(f"Municipalities: {sample['com_id'].nunique() if 'com_id' in sample.columns else 'N/A'}")
print(f"Treated municipalities (unique): {sample[sample['treated']==1]['com_id'].nunique() if 'com_id' in sample.columns and 'treated' in sample.columns else 'N/A'}")
print(f"Elections: {sample['election_year'].nunique() if 'election_year' in sample.columns else 'N/A'}")
stats = get_sample_stats(sample)
print(f"\nSample stats:")
for key, val in stats.items():
  print(f"  {key}: {val}")


## Main DiD Analysis


### Main DiD Analysis

Two-way fixed effects DiD on five political outcomes (Table 1)

In [ ]:
def run():
    """Run all main text analyses."""
    ensure_output_dirs()

    print("Loading data...")
    electoral = load_electoral_data()
    sample = prepare_analysis_sample(electoral)

    print(f"Sample size: {len(sample)} observations")
    print(f"Municipalities: {sample['com_id'].nunique()}")

    # Table 1: Main DiD
    print("\n" + "=" * 60)
    print("Table 1: Main DiD Results")
    print("=" * 60)
    table1 = run_table1_did(sample)
    print(table1)

    # Save table
    save_table_tex(
        table1,
        TABLE_DIR / "table_1.tex",
        title="Main DiD Results: Political Outcomes",
        note="TWFE DiD with municipality and year fixed effects, province-specific time trends. Standard errors clustered by Xylella infection group.",
    )

    # Figure 2a: Event study - far-right
    print("\n" + "=" * 60)
    print("Figure 2a: Event Study - Far-Right Vote Share")
    print("=" * 60)
    run_figure2a_event_study_farright(sample)
    print(f"Saved: {FIGURE_DIR / 'figure_2a.pdf'}")

    # Figure 2b: Event study - far-right/right ratio
    print("\n" + "=" * 60)
    print("Figure 2b: Event Study - Far-Right Share Within Right Bloc")
    print("=" * 60)
    run_figure2b_event_study_farright_over_right(sample)
    print(f"Saved: {FIGURE_DIR / 'figure_2b.pdf'}")

    # Figure 3a: Income null (requires income data)
    print("\n" + "=" * 60)
    print("Figure 3a: Income Null Result")
    print("=" * 60)
    try:
        income = load_income_data()
        income_sample = prepare_analysis_sample(income)
        run_figure3a_income_null(income_sample)
        print(f"Saved: {FIGURE_DIR / 'figure_3a.pdf'}")
    except Exception as e:
        print(f"Warning: Could not generate Figure 3a: {e}")

    # Figure 4: HTE by public service distance
    print("\n" + "=" * 60)
    print("Figure 4: HTE by Public Service Distance")
    print("=" * 60)
    try:
        covariates = load_covariates_data()
        run_figure4_hte_public_service(sample, covariates)
        print(f"Saved: {FIGURE_DIR / 'figure_4.pdf'}")
    except Exception as e:
        print(f"Warning: Could not generate Figure 4: {e}")

    print("\n" + "=" * 60)
    print("Main text analysis complete!")
    print("=" * 60)

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Event Study - Far-Right


### Event Study - Far-Right

Event study for far-right vote share showing parallel pre-trends (Figures 2a-2b)

In [ ]:
def run():
    """Run all main text analyses."""
    ensure_output_dirs()

    print("Loading data...")
    electoral = load_electoral_data()
    sample = prepare_analysis_sample(electoral)

    print(f"Sample size: {len(sample)} observations")
    print(f"Municipalities: {sample['com_id'].nunique()}")

    # Table 1: Main DiD
    print("\n" + "=" * 60)
    print("Table 1: Main DiD Results")
    print("=" * 60)
    table1 = run_table1_did(sample)
    print(table1)

    # Save table
    save_table_tex(
        table1,
        TABLE_DIR / "table_1.tex",
        title="Main DiD Results: Political Outcomes",
        note="TWFE DiD with municipality and year fixed effects, province-specific time trends. Standard errors clustered by Xylella infection group.",
    )

    # Figure 2a: Event study - far-right
    print("\n" + "=" * 60)
    print("Figure 2a: Event Study - Far-Right Vote Share")
    print("=" * 60)
    run_figure2a_event_study_farright(sample)
    print(f"Saved: {FIGURE_DIR / 'figure_2a.pdf'}")

    # Figure 2b: Event study - far-right/right ratio
    print("\n" + "=" * 60)
    print("Figure 2b: Event Study - Far-Right Share Within Right Bloc")
    print("=" * 60)
    run_figure2b_event_study_farright_over_right(sample)
    print(f"Saved: {FIGURE_DIR / 'figure_2b.pdf'}")

    # Figure 3a: Income null (requires income data)
    print("\n" + "=" * 60)
    print("Figure 3a: Income Null Result")
    print("=" * 60)
    try:
        income = load_income_data()
        income_sample = prepare_analysis_sample(income)
        run_figure3a_income_null(income_sample)
        print(f"Saved: {FIGURE_DIR / 'figure_3a.pdf'}")
    except Exception as e:
        print(f"Warning: Could not generate Figure 3a: {e}")

    # Figure 4: HTE by public service distance
    print("\n" + "=" * 60)
    print("Figure 4: HTE by Public Service Distance")
    print("=" * 60)
    try:
        covariates = load_covariates_data()
        run_figure4_hte_public_service(sample, covariates)
        print(f"Saved: {FIGURE_DIR / 'figure_4.pdf'}")
    except Exception as e:
        print(f"Warning: Could not generate Figure 4: {e}")

    print("\n" + "=" * 60)
    print("Main text analysis complete!")
    print("=" * 60)

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Mechanism Tests


### Mechanism Tests

Testing economic mechanism: income and young residents (Figures 3a-3b)

In [ ]:
def run():
    """Run all main text analyses."""
    ensure_output_dirs()

    print("Loading data...")
    electoral = load_electoral_data()
    sample = prepare_analysis_sample(electoral)

    print(f"Sample size: {len(sample)} observations")
    print(f"Municipalities: {sample['com_id'].nunique()}")

    # Table 1: Main DiD
    print("\n" + "=" * 60)
    print("Table 1: Main DiD Results")
    print("=" * 60)
    table1 = run_table1_did(sample)
    print(table1)

    # Save table
    save_table_tex(
        table1,
        TABLE_DIR / "table_1.tex",
        title="Main DiD Results: Political Outcomes",
        note="TWFE DiD with municipality and year fixed effects, province-specific time trends. Standard errors clustered by Xylella infection group.",
    )

    # Figure 2a: Event study - far-right
    print("\n" + "=" * 60)
    print("Figure 2a: Event Study - Far-Right Vote Share")
    print("=" * 60)
    run_figure2a_event_study_farright(sample)
    print(f"Saved: {FIGURE_DIR / 'figure_2a.pdf'}")

    # Figure 2b: Event study - far-right/right ratio
    print("\n" + "=" * 60)
    print("Figure 2b: Event Study - Far-Right Share Within Right Bloc")
    print("=" * 60)
    run_figure2b_event_study_farright_over_right(sample)
    print(f"Saved: {FIGURE_DIR / 'figure_2b.pdf'}")

    # Figure 3a: Income null (requires income data)
    print("\n" + "=" * 60)
    print("Figure 3a: Income Null Result")
    print("=" * 60)
    try:
        income = load_income_data()
        income_sample = prepare_analysis_sample(income)
        run_figure3a_income_null(income_sample)
        print(f"Saved: {FIGURE_DIR / 'figure_3a.pdf'}")
    except Exception as e:
        print(f"Warning: Could not generate Figure 3a: {e}")

    # Figure 4: HTE by public service distance
    print("\n" + "=" * 60)
    print("Figure 4: HTE by Public Service Distance")
    print("=" * 60)
    try:
        covariates = load_covariates_data()
        run_figure4_hte_public_service(sample, covariates)
        print(f"Saved: {FIGURE_DIR / 'figure_4.pdf'}")
    except Exception as e:
        print(f"Warning: Could not generate Figure 4: {e}")

    print("\n" + "=" * 60)
    print("Main text analysis complete!")
    print("=" * 60)

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Heterogeneous Effects by Public Services


### Heterogeneous Effects by Public Services

Treatment effect moderation by distance to public service hubs (Figure 4)

In [ ]:
def run():
    """Run all main text analyses."""
    ensure_output_dirs()

    print("Loading data...")
    electoral = load_electoral_data()
    sample = prepare_analysis_sample(electoral)

    print(f"Sample size: {len(sample)} observations")
    print(f"Municipalities: {sample['com_id'].nunique()}")

    # Table 1: Main DiD
    print("\n" + "=" * 60)
    print("Table 1: Main DiD Results")
    print("=" * 60)
    table1 = run_table1_did(sample)
    print(table1)

    # Save table
    save_table_tex(
        table1,
        TABLE_DIR / "table_1.tex",
        title="Main DiD Results: Political Outcomes",
        note="TWFE DiD with municipality and year fixed effects, province-specific time trends. Standard errors clustered by Xylella infection group.",
    )

    # Figure 2a: Event study - far-right
    print("\n" + "=" * 60)
    print("Figure 2a: Event Study - Far-Right Vote Share")
    print("=" * 60)
    run_figure2a_event_study_farright(sample)
    print(f"Saved: {FIGURE_DIR / 'figure_2a.pdf'}")

    # Figure 2b: Event study - far-right/right ratio
    print("\n" + "=" * 60)
    print("Figure 2b: Event Study - Far-Right Share Within Right Bloc")
    print("=" * 60)
    run_figure2b_event_study_farright_over_right(sample)
    print(f"Saved: {FIGURE_DIR / 'figure_2b.pdf'}")

    # Figure 3a: Income null (requires income data)
    print("\n" + "=" * 60)
    print("Figure 3a: Income Null Result")
    print("=" * 60)
    try:
        income = load_income_data()
        income_sample = prepare_analysis_sample(income)
        run_figure3a_income_null(income_sample)
        print(f"Saved: {FIGURE_DIR / 'figure_3a.pdf'}")
    except Exception as e:
        print(f"Warning: Could not generate Figure 3a: {e}")

    # Figure 4: HTE by public service distance
    print("\n" + "=" * 60)
    print("Figure 4: HTE by Public Service Distance")
    print("=" * 60)
    try:
        covariates = load_covariates_data()
        run_figure4_hte_public_service(sample, covariates)
        print(f"Saved: {FIGURE_DIR / 'figure_4.pdf'}")
    except Exception as e:
        print(f"Warning: Could not generate Figure 4: {e}")

    print("\n" + "=" * 60)
    print("Main text analysis complete!")
    print("=" * 60)

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Case Illustrations


### Case Illustrations

Municipality-level trajectories of far-right voting over time (Figure 5)

In [ ]:
def run():
    """Run all main text analyses."""
    ensure_output_dirs()

    print("Loading data...")
    electoral = load_electoral_data()
    sample = prepare_analysis_sample(electoral)

    print(f"Sample size: {len(sample)} observations")
    print(f"Municipalities: {sample['com_id'].nunique()}")

    # Table 1: Main DiD
    print("\n" + "=" * 60)
    print("Table 1: Main DiD Results")
    print("=" * 60)
    table1 = run_table1_did(sample)
    print(table1)

    # Save table
    save_table_tex(
        table1,
        TABLE_DIR / "table_1.tex",
        title="Main DiD Results: Political Outcomes",
        note="TWFE DiD with municipality and year fixed effects, province-specific time trends. Standard errors clustered by Xylella infection group.",
    )

    # Figure 2a: Event study - far-right
    print("\n" + "=" * 60)
    print("Figure 2a: Event Study - Far-Right Vote Share")
    print("=" * 60)
    run_figure2a_event_study_farright(sample)
    print(f"Saved: {FIGURE_DIR / 'figure_2a.pdf'}")

    # Figure 2b: Event study - far-right/right ratio
    print("\n" + "=" * 60)
    print("Figure 2b: Event Study - Far-Right Share Within Right Bloc")
    print("=" * 60)
    run_figure2b_event_study_farright_over_right(sample)
    print(f"Saved: {FIGURE_DIR / 'figure_2b.pdf'}")

    # Figure 3a: Income null (requires income data)
    print("\n" + "=" * 60)
    print("Figure 3a: Income Null Result")
    print("=" * 60)
    try:
        income = load_income_data()
        income_sample = prepare_analysis_sample(income)
        run_figure3a_income_null(income_sample)
        print(f"Saved: {FIGURE_DIR / 'figure_3a.pdf'}")
    except Exception as e:
        print(f"Warning: Could not generate Figure 3a: {e}")

    # Figure 4: HTE by public service distance
    print("\n" + "=" * 60)
    print("Figure 4: HTE by Public Service Distance")
    print("=" * 60)
    try:
        covariates = load_covariates_data()
        run_figure4_hte_public_service(sample, covariates)
        print(f"Saved: {FIGURE_DIR / 'figure_4.pdf'}")
    except Exception as e:
        print(f"Warning: Could not generate Figure 4: {e}")

    print("\n" + "=" * 60)
    print("Main text analysis complete!")
    print("=" * 60)

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Balance Table


### Balance Table

Pre-treatment balance test comparing treated vs control municipalities (Table B.19.1)

In [ ]:
def run():
    """Run high-priority appendix B analyses."""
    ensure_output_dirs()

    print("Loading data...")
    electoral = load_electoral_data()
    sample = prepare_analysis_sample(electoral)

    # Table B.19.1: Balance table
    print("\n" + "=" * 60)
    print("Table B.19.1: Balance Table")
    print("=" * 60)
    try:
        covariates = load_covariates_data()
        balance_table = run_table_b19_balance(sample, covariates)
        print(balance_table)

        save_table_tex(
            balance_table,
            TABLE_DIR / "table_b19_1.tex",
            title="Balance Table: Pre-Treatment Covariates",
            note="Two-sample t-tests comparing treated vs never-treated municipalities.",
        )
    except Exception as e:
        print(f"Warning: Could not generate Table B.19.1: {e}")

    # Figure B.7.1: Suicide event study
    print("\n" + "=" * 60)
    print("Figure B.7.1: Suicide Event Study")
    print("=" * 60)
    try:
        demographics = load_demographics_data()
        demo_sample = prepare_analysis_sample(demographics)
        run_figure_b7_suicide_event_study(demo_sample)
        print(f"Saved: {FIGURE_DIR / 'figure_b7_1.pdf'}")
    except Exception as e:
        print(f"Warning: Could not generate Figure B.7.1: {e}")

    # Table B.4.1: Dose-response
    print("\n" + "=" * 60)
    print("Table B.4.1: Dose-Response Analysis")
    print("=" * 60)
    try:
        dose_table = run_table_b4_dose_response(sample)
        print(dose_table)

        save_table_tex(
            dose_table,
            TABLE_DIR / "table_b4_1.tex",
            title="Dose-Response Analysis: Months Since Infection",
            note="TWFE models using continuous intensity (months since infection) instead of binary treatment.",
        )
    except Exception as e:
        print(f"Warning: Could not generate Table B.4.1: {e}")

    print("\n" + "=" * 60)
    print("Appendix B analysis complete!")
    print("=" * 60)

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Suicide Event Study


### Suicide Event Study

Effect of epidemic on suicide rates, testing mental health mechanism (Figure B.7.1)

In [ ]:
def run():
    """Run high-priority appendix B analyses."""
    ensure_output_dirs()

    print("Loading data...")
    electoral = load_electoral_data()
    sample = prepare_analysis_sample(electoral)

    # Table B.19.1: Balance table
    print("\n" + "=" * 60)
    print("Table B.19.1: Balance Table")
    print("=" * 60)
    try:
        covariates = load_covariates_data()
        balance_table = run_table_b19_balance(sample, covariates)
        print(balance_table)

        save_table_tex(
            balance_table,
            TABLE_DIR / "table_b19_1.tex",
            title="Balance Table: Pre-Treatment Covariates",
            note="Two-sample t-tests comparing treated vs never-treated municipalities.",
        )
    except Exception as e:
        print(f"Warning: Could not generate Table B.19.1: {e}")

    # Figure B.7.1: Suicide event study
    print("\n" + "=" * 60)
    print("Figure B.7.1: Suicide Event Study")
    print("=" * 60)
    try:
        demographics = load_demographics_data()
        demo_sample = prepare_analysis_sample(demographics)
        run_figure_b7_suicide_event_study(demo_sample)
        print(f"Saved: {FIGURE_DIR / 'figure_b7_1.pdf'}")
    except Exception as e:
        print(f"Warning: Could not generate Figure B.7.1: {e}")

    # Table B.4.1: Dose-response
    print("\n" + "=" * 60)
    print("Table B.4.1: Dose-Response Analysis")
    print("=" * 60)
    try:
        dose_table = run_table_b4_dose_response(sample)
        print(dose_table)

        save_table_tex(
            dose_table,
            TABLE_DIR / "table_b4_1.tex",
            title="Dose-Response Analysis: Months Since Infection",
            note="TWFE models using continuous intensity (months since infection) instead of binary treatment.",
        )
    except Exception as e:
        print(f"Warning: Could not generate Table B.4.1: {e}")

    print("\n" + "=" * 60)
    print("Appendix B analysis complete!")
    print("=" * 60)

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Dose-Response Analysis


### Dose-Response Analysis

Continuous treatment intensity (months since infection) instead of binary (Table B.4.1)

In [ ]:
def run():
    """Run high-priority appendix B analyses."""
    ensure_output_dirs()

    print("Loading data...")
    electoral = load_electoral_data()
    sample = prepare_analysis_sample(electoral)

    # Table B.19.1: Balance table
    print("\n" + "=" * 60)
    print("Table B.19.1: Balance Table")
    print("=" * 60)
    try:
        covariates = load_covariates_data()
        balance_table = run_table_b19_balance(sample, covariates)
        print(balance_table)

        save_table_tex(
            balance_table,
            TABLE_DIR / "table_b19_1.tex",
            title="Balance Table: Pre-Treatment Covariates",
            note="Two-sample t-tests comparing treated vs never-treated municipalities.",
        )
    except Exception as e:
        print(f"Warning: Could not generate Table B.19.1: {e}")

    # Figure B.7.1: Suicide event study
    print("\n" + "=" * 60)
    print("Figure B.7.1: Suicide Event Study")
    print("=" * 60)
    try:
        demographics = load_demographics_data()
        demo_sample = prepare_analysis_sample(demographics)
        run_figure_b7_suicide_event_study(demo_sample)
        print(f"Saved: {FIGURE_DIR / 'figure_b7_1.pdf'}")
    except Exception as e:
        print(f"Warning: Could not generate Figure B.7.1: {e}")

    # Table B.4.1: Dose-response
    print("\n" + "=" * 60)
    print("Table B.4.1: Dose-Response Analysis")
    print("=" * 60)
    try:
        dose_table = run_table_b4_dose_response(sample)
        print(dose_table)

        save_table_tex(
            dose_table,
            TABLE_DIR / "table_b4_1.tex",
            title="Dose-Response Analysis: Months Since Infection",
            note="TWFE models using continuous intensity (months since infection) instead of binary treatment.",
        )
    except Exception as e:
        print(f"Warning: Could not generate Table B.4.1: {e}")

    print("\n" + "=" * 60)
    print("Appendix B analysis complete!")
    print("=" * 60)

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))